In [1]:
import os
import pandas as pd
import glob
import csv
import cv2

# Dataset Combination

In [2]:
input_dir = r"D:\Downloads\Computer_Vision\Capstone_Project\data"
output_dir = r"D:\Downloads\Computer_Vision\Capstone_Project\data\combined_dataset"

IMAGE_DIR = os.path.join(output_dir, "images")
MASK_DIR = os.path.join(output_dir, "masks")
CSV_PATH = os.path.join(output_dir, "data_division.csv")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

In [38]:
from tqdm import tqdm

In [39]:
def process_massachusetts(writer, output_size=(1024, 1024)):
    
    mass_root = os.path.join(input_dir,'Massachusetts_Roads_Dataset', 'tiff')
    subsets = ['train', 'val', 'test']
    counter = 0

    for subset in subsets:
        img_dir = os.path.join(mass_root, subset)
        mask_dir = os.path.join(mass_root, f"{subset}_labels")

        img_paths = sorted(glob.glob(os.path.join(img_dir, "*.tiff")))
        for img_path in tqdm(img_paths):
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            mask_path = os.path.join(mask_dir, f"{base_name}.tif")
            if not os.path.exists(mask_path):
                print(f"Mask not found for {img_path}")
                continue

            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if img is None or mask is None:
                print(f"[Error] Unreadable file pair: {img_path}, {mask_path}")
                continue

            img_resized = cv2.resize(img, output_size, interpolation=cv2.INTER_LINEAR)
            mask_resized = cv2.resize(mask, output_size, interpolation=cv2.INTER_NEAREST)

            img_name = f"mass_{counter:05d}.jpg"
            mask_name = f"mass_{counter:05d}.png"

            dst_img = os.path.join(IMAGE_DIR, img_name)
            dst_mask = os.path.join(MASK_DIR, mask_name)

            cv2.imwrite(dst_img, img_resized)
            cv2.imwrite(dst_mask, mask_resized)

            writer.writerow({
                'filename': img_name,
                'maskname': mask_name,
                'split': subset
            })

            counter += 1
    print("Massachusetts processing has successfully finished!")

In [40]:
def process_deepglobe(writer, output_size=(1024, 1024)):
    dg_train = os.path.join(input_dir, 'DeepGlobe_Road_Extraction_Dataset', 'train')
    sat_imgs = sorted(glob.glob(os.path.join(dg_train, '*_sat.jpg')))
    for idx, img_path in enumerate(tqdm(sat_imgs)):
        mask_path = img_path.replace('_sat.jpg', '_mask.png')
        if not os.path.exists(mask_path):
            continue

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            print(f"[Error] Unreadable file pair: {img_path}, {mask_path}")
            continue

        img_resized = cv2.resize(img, output_size, interpolation=cv2.INTER_LINEAR)
        mask_resized = cv2.resize(mask, output_size, interpolation=cv2.INTER_NEAREST)

        img_name = f"deep_{idx:05d}.jpg"
        mask_name = f"deep_{idx:05d}.png"

        dst_img = os.path.join(IMAGE_DIR, img_name)
        dst_mask = os.path.join(MASK_DIR, mask_name)

        cv2.imwrite(dst_img, img_resized)
        cv2.imwrite(dst_mask, mask_resized)

        writer.writerow({
            'filename': img_name,
            'maskname': mask_name,
            'split': 'train'
        })
    print("DeepGlobe processing has successfully finished!")

In [41]:
with open(CSV_PATH, mode='w', newline='') as csvfile:
    fieldnames = ['filename', 'maskname', 'split']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    process_massachusetts(writer)
    process_deepglobe(writer)
    
print(f"\nDone!")

100%|██████████| 49/49 [00:02<00:00, 16.93it/s]


Massachusetts processing has successfully finished!


100%|██████████| 6226/6226 [02:36<00:00, 39.91it/s]

DeepGlobe processing has successfully finished!

Done!


# Data Division

In [6]:
df = pd.read_csv(CSV_PATH)

df = df.sample(frac=1, random_state=21).reset_index(drop=True)

n_total = len(df)
n_train = int(n_total * 0.7)
n_val = int(n_total * 0.15)

df.loc[:n_train-1, 'split'] = 'train'
df.loc[n_train:n_train+n_val-1, 'split'] = 'val'
df.loc[n_train+n_val:, 'split'] = 'test'

df.to_csv(CSV_PATH, index=False)

print(f"Division: {df['split'].value_counts().to_dict()}")

Division: {'train': 5177, 'test': 1111, 'val': 1109}
